# Extended Data Fig. 6: bipartition dependence of S_half, and the PXP scar tower

a-f: S_half of the two random networks of Extended Data Fig. 5 over all 462 balanced
bipartitions. g, h: momentum-resolved spectrum of the ring's Néel sector,
with the L + 1 scar tower of Turner et al., PRB 98, 155134 (2018).

In [ ]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qising_networks import FIGURES_DIR as FIGURES
from qising_networks import PROJECT_ROOT, spectra
from qising_networks.model import GAMMA, sector_hamiltonian
from qising_networks.plots.dynamics import example_sectors
from qising_networks.plots.figure import (
    MAX_FIG_HEIGHT,
    MAX_FIG_WIDTH,
    align_panel_labels_to_yaxis,
    journal_labels,
    save_unlabeled,
)

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

In [ ]:
systems = example_sectors()
results = [
    (name, spectra.analyse_eigenstates(*args)) for name, args in systems[:2]
]
for name, d in results:
    print(
        f"{name}: dim={d['dim']} n_atyp={d['atyp'].sum()} "
        f"r={d['r']:.3f} r_partial={d['r_partial']:.3f} "
        f"E0={d['E0']:.3f}"
    )
    print(
        f"    S_half: cut={d['half_cut']} median={np.median(d['S_half']):.3f} "
        f"r(S_half, Q)={d['r_h']:.3f} r_partial={d['r_h_partial']:.3f}"
    )
    for a in np.where(d["ent_outlier"])[0]:
        print(
            f"    S_half outlier: (E-E0)/Gamma={(d['E'][a] - d['E0']) / GAMMA:+.3f} "
            f"S_half={d['S_half'][a]:.3f} zS={d['zS'][a]:+.2f} "
            f"Q={d['Q'][a]:.3f} zQ={d['zQ'][a]:+.2f} S2-atypical={d['atyp'][a]}"
        )

In [ ]:
# PXP scar tower: the ring's Neel sector resolved by translation and
# reflection (small symmetry-breaking terms split the +-k pairs).
im_r, h_r, sec_r = systems[2][1]
N = len(h_r)
H_r = sector_hamiltonian(im_r, h_r, sec_r)
E0_r = float(np.diag(H_r).mean())
bits = (sec_r[:, None] >> (N - 1 - np.arange(N))) & 1
pos = {s: i for i, s in enumerate(sec_r)}
to_idx = lambda b: pos[int("".join(map(str, b)), 2)]
T = np.zeros_like(H_r)
T[[to_idx(np.roll(b, 1)) for b in bits], np.arange(len(sec_r))] = 1
R = np.zeros_like(H_r)
R[[to_idx(b[::-1]) for b in bits], np.arange(len(sec_r))] = 1
assert np.abs(H_r @ T - T @ H_r).max() < 1e-12
E_r, V_r = np.linalg.eigh(
    H_r + 1e-6 * (T + T.T) + 3e-7j * (T - T.T) + 2e-8 * R
)
e_r = (E_r - E0_r) / GAMMA  # PXP energy units
m_r = (
    np.rint(
        np.angle(np.einsum("ia,ij,ja->a", V_r.conj(), T, V_r))
        / (2 * np.pi / N)
    ).astype(int)
    % N
)  # momentum index
neel = [i for i, b in enumerate(bits) if all(b != np.roll(b, 1))]
ov_r = (np.abs(V_r[neel]) ** 2).sum(0)  # overlap with both Neel states
full = np.zeros((2**N, len(sec_r)), complex)
full[sec_r] = V_r
cuts_r = [[(j + i) % N for i in range(N // 2)] for j in range(N // 2)]


def half_entropy_c(full, A):
    B = [i for i in range(N) if i not in A]
    psi = (
        full.T.reshape((-1,) + (2,) * N)
        .transpose([0] + [1 + i for i in A] + [1 + i for i in B])
        .reshape(full.shape[1], 2 ** len(A), -1)
    )
    p = np.linalg.svd(psi, compute_uv=False) ** 2
    return -(p * np.log(np.where(p > 0, p, 1))).sum(1)


S_r = np.mean([half_entropy_c(full, A) for A in cuts_r], 0)

# Tower: the E = 0 state with the largest Neel overlap, then outward the
# k in {0, pi} state with the largest overlap within (0.5, 1.5) spacings
# of the previous tower state.
kzp = (m_r == 0) | (m_r == N // 2)
zero = np.where(kzp & (np.abs(e_r) < 1e-4))[0]
tower = [zero[np.argmax(ov_r[zero])]]
pos_e = np.where(kzp & (e_r > 1e-4))[0]
w0 = e_r[pos_e[np.argmax(ov_r[pos_e])]]  # first tower spacing
for sign in (1, -1):
    prev, w = 0.0, w0
    while True:
        d = sign * (e_r - prev)
        cand = np.where(kzp & (d > 0.5 * w) & (d < 1.5 * w))[0]
        if cand.size == 0:
            break
        a = cand[np.argmax(ov_r[cand])]
        tower.append(a)
        w, prev = abs(e_r[a] - prev), e_r[a]
tower = np.array(sorted(tower, key=lambda a: e_r[a]))
print("tower (E-E0)/Gamma:", np.round(e_r[tower], 3))
print("momentum index:", m_r[tower])
print(
    "S_half:",
    np.round(S_r[tower], 3),
    " sector median",
    round(np.median(S_r), 3),
)
print(
    "centre spacing",
    round(e_r[tower][N // 2 + 1] - e_r[tower][N // 2], 3),
    "vs 1.337 + 0.582/L^2 =",
    round(1.337 + 0.582 / N**2, 3),
)
print(
    "ground -> first k = pi tower state",
    round(e_r[tower][1] - e_r[tower][0], 3),
)
print("zero-energy manifold size", int((np.abs(e_r) < 1e-4).sum()))

In [ ]:
C_ALL = "0.65"  # as fig-s05.ipynb
MS = 2.0
# S_half over all balanced bipartitions: sector mean, r(S_half, Q), and
# S_half vs Q for the lowest, median and highest cut.
C_CUT = {"Lowest": "C0", "Median (used)": "k", "Highest": "C1"}
LS_CUT = {"Lowest": "-", "Median (used)": "--", "Highest": "-."}

with plt.rc_context({"font.size": 7, "legend.fontsize": 5}):
    fig_cut = plt.figure(
        figsize=(MAX_FIG_WIDTH, 0.75 * MAX_FIG_HEIGHT), constrained_layout=True
    )
    grid_cut = fig_cut.subplots(4, 2)
    for (ax_m, ax_r, ax_q, _), (name, d) in zip(grid_cut.T, results):
        S, cuts = d["all_cuts"]
        mean = S.mean(1)
        r_cut = np.array([np.corrcoef(s, d["Q"])[0, 1] for s in S])
        order = np.argsort(mean)
        picks = dict(zip(C_CUT, (order[0], order[len(cuts) // 2], order[-1])))
        print(
            f"{name}: {len(cuts)} cuts, sector mean {mean.min():.3f}-"
            f"{mean.max():.3f}, r(S_half, Q) {r_cut.min():.3f}-{r_cut.max():.3f}"
        )
        for lab, k in picks.items():
            print(
                f"    {lab}: cut={cuts[k]} mean={mean[k]:.3f} r={r_cut[k]:.3f}"
            )

        ax_m.set_title(name)
        ax_m.hist(mean, bins=30, color=C_ALL)
        ax_r.hist(r_cut, bins=30, color=C_ALL)
        for lab, k in picks.items():
            ax_m.axvline(mean[k], color="k", ls=LS_CUT[lab], lw=0.7, label=lab)
            ax_r.axvline(r_cut[k], color="k", ls=LS_CUT[lab], lw=0.7)
            ax_q.plot(
                d["Q"],
                S[k],
                "o",
                ms=MS - 0.8,
                mec="none",
                color=C_CUT[lab],
                alpha=0.6,
                rasterized=True,
                label=f"{lab} ($r = {r_cut[k]:.3f}$)",
            )
        ax_m.set_xlabel(r"Sector mean of $S_\mathrm{half}$")
        ax_r.set_xlabel(r"$r(S_\mathrm{half}, Q)$")
        ax_m.set_ylabel("Number of\nbipartitions")
        ax_r.set_ylabel("Number of\nbipartitions")
        ax_q.set_xlabel(r"Multipartite entanglement $Q$")
        ax_q.set_ylabel("Entanglement\nentropy $S_\\mathrm{half}$")
        ax_q.legend(
            frameon=False, loc="upper left", markerscale=1.5, handletextpad=0.3
        )
    grid_cut[0, 0].legend(
        frameon=False,
        loc="upper left",
        handlelength=3,
        bbox_to_anchor=(0.06, 1.0),
    )  # clear of the lowest line

    # g, h: PXP scar tower, k = 0 and k = pi tower states highlighted.
    ax_s, ax_o = grid_cut[3]
    is0 = m_r[tower] == 0
    for ax, y in ((ax_s, S_r), (ax_o, np.log10(ov_r))):
        ax.plot(
            e_r,
            y,
            "o",
            ms=MS,
            mec="none",
            color=C_ALL,
            rasterized=True,
            label="Eigenstates",
        )
        ax.plot(
            e_r[tower[is0]],
            y[tower[is0]],
            "o",
            ms=MS + 0.5,
            mec="none",
            color="C3",
            label=r"Scar tower, $\kappa = 0$",
        )
        ax.plot(
            e_r[tower[~is0]],
            y[tower[~is0]],
            "s",
            ms=MS + 0.3,
            mec="none",
            color="C0",
            label=r"Scar tower, $\kappa = \pi$",
        )
        ax.set_xlabel(r"$(E - E_0)/\mathit{\Gamma}$")
    ax_s.set_ylabel("Entanglement\nentropy $S_\\mathrm{half}$")
    ax_o.set_ylabel(r"$\log_{10}|\langle\mathbb{Z}_2|E\rangle|^2$")
    ax_o.set_ylim(bottom=-8)
    ax_s.set_title(
        "PXP Ising chain (crystal momentum resolved, no random field)",
        loc="left",
    )
    ax_s.legend(
        frameon=False, loc="lower center", markerscale=1.5, handletextpad=0.3
    )
    axes_cut = list(grid_cut.ravel())
    align_panel_labels_to_yaxis(
        fig_cut, axes_cut, journal_labels("abcdefgh"), fontsize=8, bold=True
    )

In [ ]:
paths = save_unlabeled(fig_cut, FIGURES / "fig-s06", "fig-s06")
print(paths)